In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

In [2]:
transform = transforms.ToTensor()

dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

In [3]:
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size]
)

In [4]:
batch_size = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [5]:
class MLP(nn.Module):

    def __init__(self, dropout_rate=0.3):
        super().__init__()

        self.network = nn.Sequential(

            nn.Flatten(),

            nn.Linear(28 * 28, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.network(x)

In [6]:
nn.Dropout(0.3)

Dropout(p=0.3, inplace=False)

In [7]:
nn.BatchNorm1d(256)

BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)

In [8]:
criterion = nn.CrossEntropyLoss()

In [10]:
def train_one_epoch(model, loader, criterion, optimizer, device):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(loader)
    accuracy = correct / total

    return avg_loss, accuracy

In [14]:
def validate(model, loader, criterion, device):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            total_loss += loss.item()

            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(loader)
    accuracy = correct / total

    return avg_loss, accuracy

In [15]:
class EarlyStopping:

    def __init__(self, patience=3):

        self.patience = patience
        self.counter = 0
        self.best_loss = float("inf")
        self.best_model = None

    def step(self, val_loss, model):

        if val_loss < self.best_loss:

            self.best_loss = val_loss
            self.counter = 0

            self.best_model = {
                key: value.cpu().clone()
                for key, value in model.state_dict().items()
            }

        else:

            self.counter += 1

        return self.counter >= self.patience

In [16]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

model = MLP(dropout_rate=0.3).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

early_stopping = EarlyStopping(patience=3)

epochs = 20

for epoch in range(epochs):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    val_loss, val_acc = validate(
        model,
        val_loader,
        criterion,
        device
    )

    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_acc:.4f} "
        f"Val Loss: {val_loss:.4f} "
        f"Val Acc: {val_acc:.4f} "
        f"LR: {current_lr:.6f}"
    )

    if early_stopping.step(val_loss, model):

        print("Early stopping triggered.")

        break

Using device: cpu
Epoch [1/20] Train Loss: 0.3257 Train Acc: 0.9087 Val Loss: 0.1292 Val Acc: 0.9627 LR: 0.001000
Epoch [2/20] Train Loss: 0.1643 Train Acc: 0.9497 Val Loss: 0.1061 Val Acc: 0.9667 LR: 0.001000
Epoch [3/20] Train Loss: 0.1280 Train Acc: 0.9600 Val Loss: 0.0939 Val Acc: 0.9713 LR: 0.001000
Epoch [4/20] Train Loss: 0.1121 Train Acc: 0.9647 Val Loss: 0.0911 Val Acc: 0.9722 LR: 0.001000
Epoch [5/20] Train Loss: 0.0987 Train Acc: 0.9685 Val Loss: 0.0815 Val Acc: 0.9766 LR: 0.001000
Epoch [6/20] Train Loss: 0.0914 Train Acc: 0.9709 Val Loss: 0.0805 Val Acc: 0.9756 LR: 0.001000
Epoch [7/20] Train Loss: 0.0854 Train Acc: 0.9731 Val Loss: 0.0764 Val Acc: 0.9770 LR: 0.001000
Epoch [8/20] Train Loss: 0.0789 Train Acc: 0.9748 Val Loss: 0.0737 Val Acc: 0.9788 LR: 0.001000
Epoch [9/20] Train Loss: 0.0711 Train Acc: 0.9772 Val Loss: 0.0763 Val Acc: 0.9783 LR: 0.001000
Epoch [10/20] Train Loss: 0.0705 Train Acc: 0.9775 Val Loss: 0.0783 Val Acc: 0.9772 LR: 0.001000
Epoch [11/20] Train L

In [17]:
model.load_state_dict(early_stopping.best_model)

<All keys matched successfully>

In [18]:
learning_rates = [
    0.01,
    0.001,
    0.0001
]

In [19]:
def run_experiment(lr):

    model = MLP(dropout_rate=0.3).to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=1e-4
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=2
    )

    early_stopping = EarlyStopping(patience=3)

    best_val_loss = float("inf")
    best_val_acc = 0

    for epoch in range(20):

        train_loss, train_acc = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            device
        )

        val_loss, val_acc = validate(
            model,
            val_loader,
            criterion,
            device
        )

        scheduler.step(val_loss)

        if val_loss < best_val_loss:

            best_val_loss = val_loss
            best_val_acc = val_acc

        if early_stopping.step(val_loss, model):

            break

    return best_val_loss, best_val_acc

In [20]:
results = {}

learning_rates = [
    0.01,
    0.001,
    0.0001
]

for lr in learning_rates:

    print(f"\nTraining with learning rate = {lr}")

    val_loss, val_acc = run_experiment(lr)

    results[lr] = {
        "val_loss": val_loss,
        "val_accuracy": val_acc
    }


Training with learning rate = 0.01

Training with learning rate = 0.001

Training with learning rate = 0.0001


In [21]:
print("\nLearning Rate Results")

for lr, result in results.items():

    print(
        f"LR: {lr} | "
        f"Val Loss: {result['val_loss']:.4f} | "
        f"Val Accuracy: {result['val_accuracy']:.4f}"
    )


Learning Rate Results
LR: 0.01 | Val Loss: 0.1261 | Val Accuracy: 0.9612
LR: 0.001 | Val Loss: 0.0730 | Val Accuracy: 0.9788
LR: 0.0001 | Val Loss: 0.0744 | Val Accuracy: 0.9772


In [22]:
best_lr = min(
    results,
    key=lambda lr: results[lr]["val_loss"]
)

print("\nBest learning rate:", best_lr)


Best learning rate: 0.001


In [23]:
best_lr = max(
    results,
    key=lambda lr: results[lr]["val_accuracy"]
)

print("Best learning rate:", best_lr)

Best learning rate: 0.001


In [24]:
model = MLP(
    dropout_rate=0.3
)

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

early_stopping = EarlyStopping(
    patience=3
)